In [1]:
# ============================================================
# AI Campaign Experimentation Analyzer
# Author: Suyash Kashyap
# Purpose:
# Automate A/B test analysis for marketing campaign creatives
# ============================================================

In [2]:
# Core libraries for data analysis

import pandas as pd
import numpy as np

# Statistical testing library
from statsmodels.stats.proportion import proportions_ztest

In [3]:
# ------------------------------------------------------------
# Generate synthetic marketing campaign test data
# ------------------------------------------------------------

np.random.seed(42)

creatives = ["Creative_A","Creative_B","Creative_C"]
segments = ["Segment_1","Segment_2","Segment_3"]
subsegments = ["Sub_1","Sub_2"]
test_control = ["Test","Control"]

rows = []

for i in range(300):

    creative = np.random.choice(creatives)
    segment = np.random.choice(segments)
    subsegment = np.random.choice(subsegments)
    tag = np.random.choice(test_control)

    snd = np.random.randint(8000,15000)

    open_rate = np.random.uniform(0.22,0.35)
    click_rate = np.random.uniform(0.08,0.18)
    offer_rate = np.random.uniform(0.30,0.50)
    app_rate = np.random.uniform(0.60,0.85)
    listing_rate = np.random.uniform(0.70,0.95)

    opn = int(snd * open_rate)
    clk = int(opn * click_rate)

    offer = int(clk * offer_rate)
    app = int(offer * app_rate)

    listings = int(app * listing_rate)

    avg_loan = np.random.randint(20000,40000)

    listing_amount = listings * avg_loan

    unsub = int(snd * np.random.uniform(0.002,0.01))

    rows.append([
        creative,
        segment,
        subsegment,
        tag,
        snd,
        opn,
        clk,
        offer,
        app,
        listings,
        listing_amount,
        unsub
    ])

df = pd.DataFrame(rows, columns=[
"Creative_name",
"Segment",
"Subsegment",
"Test_control_tagging",
"snd",
"opn",
"clk",
"offer",
"app",
"no_of_listings",
"Listing_amount",
"Unsub"
])

df.head()

,Creative_name,Segment,Subsegment,Test_control_tagging,snd,opn,clk,offer,app,no_of_listings,Listing_amount,Unsub
0,Creative_C,Segment_1,Sub_1,Test,13191,3928,375,124,76,69,2164047,41
1,Creative_C,Segment_2,Sub_1,Control,10391,3410,345,116,74,57,1504572,56
2,Creative_A,Segment_3,Sub_1,Test,10047,2591,302,118,93,69,1467423,67
3,Creative_C,Segment_1,Sub_1,Test,14164,3945,320,156,115,91,2057692,54
4,Creative_C,Segment_3,Sub_2,Test,8775,1969,336,118,90,70,2215710,32


In [4]:
# ------------------------------------------------------------
# Calculate campaign funnel metrics
# ------------------------------------------------------------

df["Open_Rate"] = df["opn"] / df["snd"]

df["Click_Rate"] = df["clk"] / df["snd"]

df["App_rate"] = df["app"] / df["clk"]

df["Response_rate"] = df["app"] / df["snd"]

df["Offer_rate"] = df["offer"] / df["app"]

df["Take_rate"] = df["no_of_listings"] / df["offer"]

df["LPM"] = df["Listing_amount"] / df["snd"]

df["ALS"] = df["Listing_amount"] / df["no_of_listings"]

df["Unsub_rate"] = df["Unsub"] / df["snd"]

df.head()

,Creative_name,Segment,Subsegment,Test_control_tagging,snd,opn,clk,offer,app,no_of_listings,...,Unsub,Open_Rate,Click_Rate,App_rate,Response_rate,Offer_rate,Take_rate,LPM,ALS,Unsub_rate
0,Creative_C,Segment_1,Sub_1,Test,13191,3928,375,124,76,69,...,41,0.297779,0.028428,0.202667,0.005762,1.631579,0.556452,164.054810,31363.0,0.003108
1,Creative_C,Segment_2,Sub_1,Control,10391,3410,345,116,74,57,...,56,0.328169,0.033202,0.214493,0.007122,1.567568,0.491379,144.795689,26396.0,0.005389
2,Creative_A,Segment_3,Sub_1,Test,10047,2591,302,118,93,69,...,67,0.257888,0.030059,0.307947,0.009256,1.268817,0.584746,146.055838,21267.0,0.006669
3,Creative_C,Segment_1,Sub_1,Test,14164,3945,320,156,115,91,...,54,0.278523,0.022592,0.359375,0.008119,1.356522,0.583333,145.276193,22612.0,0.003812
4,Creative_C,Segment_3,Sub_2,Test,8775,1969,336,118,90,70,...,32,0.224387,0.038291,0.267857,0.010256,1.311111,0.593220,252.502564,31653.0,0.003647


In [5]:
# ------------------------------------------------------------
# Detect winning creatives using statistical significance
# ------------------------------------------------------------

creative_summary = df.groupby(
    ["Creative_name","Test_control_tagging"]
).agg({
    "snd":"sum",
    "app":"sum"
}).reset_index()

results = []

for creative in df["Creative_name"].unique():

    subset = creative_summary[
        creative_summary["Creative_name"] == creative
    ]

    if len(subset) == 2:

        test_row = subset[
            subset["Test_control_tagging"]=="Test"
        ]

        control_row = subset[
            subset["Test_control_tagging"]=="Control"
        ]

        success = [
            int(test_row["app"]),
            int(control_row["app"])
        ]

        samples = [
            int(test_row["snd"]),
            int(control_row["snd"])
        ]

        z_stat, p_value = proportions_ztest(success, samples)

        test_rate = success[0]/samples[0]
        control_rate = success[1]/samples[1]

        if p_value < 0.10:
            winner = "Test" if test_rate > control_rate else "Control"
        else:
            winner = "No Winner"

        results.append([
            creative,
            test_rate,
            control_rate,
            p_value,
            winner
        ])

results_df = pd.DataFrame(results,columns=[
    "Creative",
    "Test_Response_Rate",
    "Control_Response_Rate",
    "P_value",
    "Winner"
])

results_df

,Creative,Test_Response_Rate,Control_Response_Rate,P_value,Winner
0,Creative_C,0.011338,0.010265,5.027317e-08,Test
1,Creative_A,0.011065,0.010429,7.631815e-04,Test
2,Creative_B,0.010203,0.010626,2.440009e-02,Control


In [6]:
# ------------------------------------------------------------
# Detect which segment shows strongest lift
# ------------------------------------------------------------

segment_summary = df.groupby(
    ["Segment","Test_control_tagging"]
).agg({
    "snd":"sum",
    "app":"sum"
}).reset_index()

segment_results = []

for seg in df["Segment"].unique():

    subset = segment_summary[
        segment_summary["Segment"] == seg
    ]

    if len(subset) == 2:

        test_row = subset[
            subset["Test_control_tagging"]=="Test"
        ]

        control_row = subset[
            subset["Test_control_tagging"]=="Control"
        ]

        success = [
            int(test_row["app"]),
            int(control_row["app"])
        ]

        samples = [
            int(test_row["snd"]),
            int(control_row["snd"])
        ]

        z_stat, p_value = proportions_ztest(success, samples)

        test_rate = success[0]/samples[0]
        control_rate = success[1]/samples[1]

        if p_value < 0.10:
            winner = "Test" if test_rate > control_rate else "Control"
        else:
            winner = "No Winner"

        segment_results.append([
            seg,
            test_rate,
            control_rate,
            p_value,
            winner
        ])

segment_results_df = pd.DataFrame(segment_results,columns=[
    "Segment",
    "Test_Response_Rate",
    "Control_Response_Rate",
    "P_value",
    "Winner"
])

segment_results_df

,Segment,Test_Response_Rate,Control_Response_Rate,P_value,Winner
0,Segment_1,0.010467,0.010315,0.452688,No Winner
1,Segment_2,0.010881,0.010796,0.645539,No Winner
2,Segment_3,0.011072,0.010188,0.000002,Test


In [7]:
# ------------------------------------------------------------
# Detect which segment shows strongest lift
# ------------------------------------------------------------

segment_summary = df.groupby(
    ["Segment","Test_control_tagging"]
).agg({
    "snd":"sum",
    "app":"sum"
}).reset_index()

segment_results = []

for seg in df["Segment"].unique():

    subset = segment_summary[
        segment_summary["Segment"] == seg
    ]

    if len(subset) == 2:

        test_row = subset[
            subset["Test_control_tagging"]=="Test"
        ]

        control_row = subset[
            subset["Test_control_tagging"]=="Control"
        ]

        success = [
            int(test_row["app"]),
            int(control_row["app"])
        ]

        samples = [
            int(test_row["snd"]),
            int(control_row["snd"])
        ]

        z_stat, p_value = proportions_ztest(success, samples)

        test_rate = success[0]/samples[0]
        control_rate = success[1]/samples[1]

        if p_value < 0.10:
            winner = "Test" if test_rate > control_rate else "Control"
        else:
            winner = "No Winner"

        segment_results.append([
            seg,
            test_rate,
            control_rate,
            p_value,
            winner
        ])

segment_results_df = pd.DataFrame(segment_results,columns=[
    "Segment",
    "Test_Response_Rate",
    "Control_Response_Rate",
    "P_value",
    "Winner"
])

segment_results_df

,Segment,Test_Response_Rate,Control_Response_Rate,P_value,Winner
0,Segment_1,0.010467,0.010315,0.452688,No Winner
1,Segment_2,0.010881,0.010796,0.645539,No Winner
2,Segment_3,0.011072,0.010188,0.000002,Test


In [8]:
# ------------------------------------------------------------
# Generate automated experiment insights
# This simulates an "AI analyst"
# ------------------------------------------------------------

insights = []

# Overall creative winner

winner_creatives = results_df[
    results_df["Winner"]=="Test"
]["Creative"].tolist()

if len(winner_creatives) > 0:

    insights.append(
        f"Winning creative(s): {', '.join(winner_creatives)} "
        "show statistically significant lift at 90% confidence."
    )

else:

    insights.append(
        "No creative shows statistically significant improvement."
    )


# Best performing segment

best_segment = segment_results_df.sort_values(
    "Test_Response_Rate",
    ascending=False
).iloc[0]["Segment"]

insights.append(
    f"Segment with highest response rate: {best_segment}"
)


# Recommendation

if len(winner_creatives) > 0:

    insights.append(
        "Recommendation: Scale winning creative to high performing segments."
    )

else:

    insights.append(
        "Recommendation: Continue testing new creative variants."
    )


for i in insights:
    print("•", i)

• Winning creative(s): Creative_C, Creative_A show statistically significant lift at 90% confidence.
• Segment with highest response rate: Segment_3
• Recommendation: Scale winning creative to high performing segments.


In [13]:
# Advanced AI-like Campaign Analytics Copilot (No API Needed)

In [15]:
# ============================================================
# Advanced AI-like Campaign Analytics Copilot (No API Needed)
# ============================================================

def ask_campaign_ai(question, results_df, segment_results_df, df):

    q = question.lower()

    # ------------------------------
    # Creative winner detection
    # ------------------------------

    if "creative" in q and ("win" in q or "won" in q or "winner" in q):

        winners = results_df[results_df["Winner"]=="Test"]["Creative"].tolist()

        if winners:
            return f"Winning creative(s): {', '.join(winners)} show statistically significant improvement at 90% confidence."
        else:
            return "No creative shows statistically significant improvement versus control."

    # ------------------------------
    # Best segment
    # ------------------------------

    if "segment" in q and ("best" in q or "perform" in q or "strong" in q):

        best_segment = segment_results_df.sort_values(
            "Test_Response_Rate",
            ascending=False
        ).iloc[0]["Segment"]

        return f"{best_segment} shows the strongest response rate improvement."

    # ------------------------------
    # Statistical significance
    # ------------------------------

    if "significant" in q or "statistical" in q:

        sig_creatives = results_df[results_df["Winner"]=="Test"]

        if len(sig_creatives) > 0:

            names = sig_creatives["Creative"].tolist()

            return f"Statistically significant improvement detected for: {', '.join(names)}."

        else:

            return "No statistically significant improvement detected at 90% confidence."

    # ------------------------------
    # Response rate question
    # ------------------------------

    if "response rate" in q:

        avg_rate = df["Response_rate"].mean()

        return f"Average campaign response rate is {avg_rate:.4f}."

    # ------------------------------
    # KPI improvement
    # ------------------------------

    if "kpi" in q or "metric" in q or "improve" in q:

        metrics = [
            "Open_Rate",
            "Click_Rate",
            "Response_rate",
            "Offer_rate",
            "Take_rate"
        ]

        improvements = {}

        for m in metrics:
            improvements[m] = df[m].mean()

        best_kpi = max(improvements, key=improvements.get)

        return f"{best_kpi} shows the strongest overall performance."

    # ------------------------------
    # Recommendation logic
    # ------------------------------

    if "recommend" in q or "scale" in q or "next" in q:

        winners = results_df[results_df["Winner"]=="Test"]["Creative"].tolist()

        best_segment = segment_results_df.sort_values(
            "Test_Response_Rate",
            ascending=False
        ).iloc[0]["Segment"]

        if winners:
            return f"Recommendation: Scale {', '.join(winners)} to {best_segment} where response rate is strongest."
        else:
            return "Recommendation: Continue testing new creative variants."

    # ------------------------------
    # Funnel insight
    # ------------------------------

    if "funnel" in q:

        return f"""
Campaign Funnel Overview

Send → Open → Click → Offer → Application → Listings

Average Metrics:
Open Rate: {df['Open_Rate'].mean():.3f}
Click Rate: {df['Click_Rate'].mean():.3f}
Response Rate: {df['Response_rate'].mean():.3f}
Take Rate: {df['Take_rate'].mean():.3f}
"""

    # ------------------------------
    # Default response
    # ------------------------------

    return """
I can answer questions about:

• Creative winners
• Segment performance
• Statistical significance
• KPI improvements
• Campaign funnel
• Experiment recommendations
"""

In [16]:
question = input("Ask a question about the campaign experiment: ")

answer = ask_campaign_ai(
    question,
    results_df,
    segment_results_df,
    df
)

print("\nAI Analyst:")
print(answer)

Ask a question about the campaign experiment: Is the test statistically significant?

AI Analyst:
Statistically significant improvement detected for: Creative_C, Creative_A.
